# Fake News Detection

### Load Dataset

In [1]:
import pandas as pd
df_true = pd.read_csv('True.csv')
df_fake = pd.read_csv('Fake.csv')

### Add labels: 1 for real, 0 for fake

In [2]:
df_true["label"] = 1
df_fake["label"] = 0

### Combine and Shuffle

#### Combine

In [3]:
df = pd.concat([df_true,df_fake])
#df.to_csv("combined_data.csv", index=False)

#### Shuffle


In [4]:
# Gives 50% data randomly
#df_fifty_percent = df.sample(frac= .5)

In [5]:
df = df.sample(frac= 1).reset_index(drop=True)
#df.to_csv("shuffled_combined_data.csv", index=False)

In [6]:
#df_confusion = df.sample(frac= 0.001).reset_index(drop=True)

### Splitting the dataset into the training set and the test set


In [7]:
short_combined_data = pd.read_csv("short_combined_data.csv")

In [39]:
from sklearn.model_selection import train_test_split
x=df["text"]
y=df["label"]
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

### Exploiting the TfidfVectorizer

In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer
# stop_word = "english" -> extract the common words like(the,is,am,are)
# max_df = maximum document frequency, if the word is present in the 70% of each document. then is will be a common word and exclude the word by the command. those are the noisy word.
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df = 0.7)


#### Fitting and Trasforming the data

In [41]:
tfidf_x_train = tfidf_vectorizer.fit_transform(x_train)
tfidf_x_test = tfidf_vectorizer.transform(x_test)

## Model Training (Passive Aggresive Classifier)

In [43]:
from sklearn.linear_model import PassiveAggressiveClassifier
pac = PassiveAggressiveClassifier(max_iter=50)


In [44]:
pac.fit(tfidf_x_train,y_train)

PassiveAggressiveClassifier(max_iter=50)

## Predicting the test set result

In [51]:
print(x_test)

22216    WASHINGTON (Reuters) - The White House said on...
27917    We all remember the absolutely disastrous visi...
25007    HANOVER, Germany (Reuters) - German Chancellor...
1377     BERLIN (Reuters) - The U.N. Special Envoy for ...
32476    After 18 months of rampant speculation over Tr...
                               ...                        
42119    The co-founders of Ben & Jerry s Ice Cream, Be...
4068     (Reuters) - The chief executives of Intel Corp...
22498    Maybe the  Queen of Incompetence  isn t as pop...
14658                                                     
15236    No reason to deal in facts. A black man was ki...
Name: text, Length: 8980, dtype: object


In [52]:
print(y_test)

22216    1
27917    0
25007    1
1377     1
32476    0
        ..
42119    0
4068     1
22498    0
14658    0
15236    0
Name: label, Length: 8980, dtype: int64


In [47]:
y_pred = pac.predict(tfidf_x_test)

In [53]:
y_pred

array([1, 0, 1, ..., 0, 0, 0])

## Model Evaluation

In [49]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test,y_pred)

array([[4744,   37],
       [  23, 4176]])

In [50]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.9933184855233853

## Testing with your own news

In [73]:
def test_news(text):
  data = tfidf_vectorizer.transform([text])
  prediction = pac.predict(data)
  return "The news is real" if prediction == 1 else "The news is fake"


In [75]:
#False
test_news("Breaking: Scientist discovered that the moon is actually made of green cheese")

'The news is fake'

In [76]:
#True
test_news("Unvaccinated tourists will not be allowed into Canada for some time -PM Trudeau")

'The news is fake'

In [77]:
#True
test_news("Brazil's Bolsonaro says he may not accept 2022 election under current voting system")

'The news is real'

In [78]:
#True
test_news("Chinese city building new quarantine centre as Covid-19 cases rise")

'The news is fake'

In [79]:
#True
test_news("Putin cites ills in US society after Biden’s killer remark")

'The news is fake'

In [80]:
#True
test_news("Minneapolis judge bars DHS agents from arresting peaceful protesters")

'The news is fake'